# Assignment 2 -- Part 2: DataFrame / Spark ML Pipeline

Reproduces the Assignment 1 selection step using **only Spark built-in transformers**:

1. `RegexTokenizer` -- unigram tokenisation, splitting on whitespace, tabs, digits and the delimiters `()[]{}.!?,;:+=-_"'`~#@&*%€$§\/` (case-folding via `toLowercase`).
2. `StopWordsRemover` -- drops the assignment stopword list.
3. `CountVectorizer` + `IDF` -- TF-IDF feature vectors.
4. `StringIndexer` on the category label.
5. `ChiSqSelector` with `numTopFeatures=2000` -- 2000 top terms overall by chi-square.

Output: `output_ds.txt` listing the 2000 selected terms in alphabetical order (matches the joined-dictionary line of Assignment 1).

In [1]:
from __future__ import annotations

from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    StopWordsRemover,
    CountVectorizer,
    IDF,
    StringIndexer,
    ChiSqSelector,
)

## Configuration

Same paths as Part 1 -- `INPUT_PATH` can be repointed at HDFS on the cluster.

In [2]:
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

INPUT_PATH = str(REPO_ROOT / "data" / "reviews_devset.json")
STOPWORDS_PATH = str(REPO_ROOT / "data" / "stopwords.txt")
OUTPUT_PATH = str(REPO_ROOT / "output_ds.txt")
TOP_N = 2000

INPUT_PATH, STOPWORDS_PATH, OUTPUT_PATH

('/Users/martinweber/Development/Personal/data-intensive-computing/data/reviews_devset.json',
 '/Users/martinweber/Development/Personal/data-intensive-computing/data/stopwords.txt',
 '/Users/martinweber/Development/Personal/data-intensive-computing/output_ds.txt')

In [3]:
spark = (
    SparkSession.builder
    .appName("assignment2-part2-dataframe")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

26/05/08 11:06:25 WARN Utils: Your hostname, MacBook-Pro-von-Martin-2.local resolves to a loopback address: 127.0.0.1; using 10.0.0.3 instead (on interface en0)
26/05/08 11:06:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 11:06:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load reviews

One JSON object per line. Keep the columns the pipeline needs and drop rows missing either field.

In [4]:
reviews = (
    spark.read.json(INPUT_PATH)
         .select("category", F.coalesce(F.col("reviewText"), F.lit("")).alias("text"))
         .where(F.col("category").isNotNull())
)
reviews.cache()
reviews.count()

78829

## Stopwords

Loaded from the assignment file rather than the default Spark list so results are comparable to Part 1.

In [5]:
with open(STOPWORDS_PATH, encoding="utf-8") as fh:
    stopwords = [line.strip() for line in fh if line.strip()]
len(stopwords)

596

## Pipeline

`RegexTokenizer` with `gaps=True` treats the regex as a *delimiter* pattern, so the character class below is exactly the Assignment 1 split set: whitespace (`\s` covers spaces and tabs), digits (`\d`), and the punctuation delimiters. `toLowercase=True` performs the case-folding step. `minTokenLength=2` drops single-character tokens to mirror Part 1.

In [6]:
DELIMITER_REGEX = r"[\s\d()\[\]{}.!?,;:+=_\"'`~#@&*%\u20ac$\u00a7\\/-]+"

tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="tokens_raw",
    pattern=DELIMITER_REGEX,
    gaps=True,
    toLowercase=True,
    minTokenLength=2,
)

remover = StopWordsRemover(
    inputCol="tokens_raw",
    outputCol="tokens",
    stopWords=stopwords,
    caseSensitive=False,
)

vectorizer = CountVectorizer(
    inputCol="tokens",
    outputCol="tf",
)

idf = IDF(inputCol="tf", outputCol="features")

label_indexer = StringIndexer(
    inputCol="category",
    outputCol="label",
    handleInvalid="skip",
)

selector = ChiSqSelector(
    numTopFeatures=TOP_N,
    featuresCol="features",
    labelCol="label",
    outputCol="selectedFeatures",
)

pipeline = Pipeline(stages=[tokenizer, remover, vectorizer, idf, label_indexer, selector])

## Fit

Fitting the pipeline trains the vocabulary, IDF weights, the label index and the chi-square selector in one pass each.

In [7]:
model = pipeline.fit(reviews)
cv_model = model.stages[2]
selector_model = model.stages[5]
len(cv_model.vocabulary), len(selector_model.selectedFeatures)

26/05/08 11:06:36 WARN DAGScheduler: Broadcasting large task binary with size 1074.1 KiB


26/05/08 11:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1075.1 KiB


26/05/08 11:06:40 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/05/08 11:06:40 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB


26/05/08 11:06:41 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB


(96125, 2000)

## Selected terms

`ChiSqSelector` exposes `selectedFeatures` -- the indices it kept from the input vector. Map them back through the `CountVectorizer` vocabulary and sort alphabetically for the output file.

In [8]:
vocab = cv_model.vocabulary
selected_terms = sorted(vocab[i] for i in selector_model.selectedFeatures)
len(selected_terms), selected_terms[:10]

(2000,
 ['access',
  'accessories',
  'account',
  'acid',
  'acoustic',
  'act',
  'acted',
  'acting',
  'action',
  'actions'])

## Write `output_ds.txt`

Single line: the 2000 selected terms separated by spaces.

In [9]:
Path(OUTPUT_PATH).write_text(" ".join(selected_terms) + "\n", encoding="utf-8")
print(f"Wrote {OUTPUT_PATH} ({len(selected_terms)} terms).")

Wrote /Users/martinweber/Development/Personal/data-intensive-computing/output_ds.txt (2000 terms).


In [10]:
reviews.unpersist()
spark.stop()